# ПР3. Пайплайн предобработки данных

**Цель:** собрать воспроизводимый пайплайн предобработки, исключающий утечку
данных из test в train.

**Основания для выбора стратегий** — отчёт о проблемах данных из ПР-2:
- дублирующиеся признаки (corr ≈ 1) удаляются;
- пропуски в числовых признаках заполняются медианой (устойчива к выбросам,
  которые обнаружены в ПР-2: max diameter 939 км, moid 79.5 а.е.);
- пропуски в категориальных — отдельной категорией (через most_frequent
  нельзя: смешивает смысл; используем константу "unknown");
- масштабирование StandardScaler — нужно для линейных моделей (ПР4);
- все статистики считаются только на train.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/dataset.csv", low_memory=False)

drop_cols = [
    # идентификаторы и служебные поля
    "id", "spkid", "full_name", "pdes", "name", "prefix",
    "orbit_id",   # ID версии орбитального решения: тысячи уникальных значений, мусор для OneHot
    "equinox",    # константа (J2000) — нулевая информативность
    # дубли (выявлены в ПР-2, corr ≈ 1)
    "moid_ld", "epoch_mjd", "epoch_cal", "per_y", "tp_cal",
    # погрешность целевой переменной — прокси-утечка
    "diameter_sigma",
]

df_model = df.drop(columns=drop_cols)
print("Осталось признаков:", df_model.shape[1])

Осталось признаков: 31


In [2]:
# ЗАДАЧА 1: классификация pha
# diameter исключаем: 86% пропусков, медиана на 800k объектов = фантазия (отчёт ПР-2, проблема №1)
clf_data = df_model.dropna(subset=["pha"]).copy()
X_clf = clf_data.drop(columns=["pha", "diameter"])
y_clf = clf_data["pha"]

# ЗАДАЧА 2: регрессия diameter (подвыборка с известным диаметром)
# pha оставляем как признак: известен почти для всех объектов, информативен (ПР-2)
reg_data = df_model.dropna(subset=["diameter"]).copy()
X_reg = reg_data.drop(columns=["diameter"])
y_reg = reg_data["diameter"]

print("Классификация:", X_clf.shape, "| регрессия:", X_reg.shape)

Классификация: (938603, 29) | регрессия: (136209, 30)


In [3]:
from sklearn.model_selection import train_test_split

# Разбиение выполняется ДО расчёта любых статистик — ключевое условие
# отсутствия утечки (требование ПР-3)
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print("CLF train/test:", X_clf_train.shape, X_clf_test.shape)
print("REG train/test:", X_reg_train.shape, X_reg_test.shape)

CLF train/test: (750882, 29) (187721, 29)
REG train/test: (108967, 30) (27242, 30)


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Числовые и категориальные признаки
num_cols = X_clf_train.select_dtypes(include=["float64", "int64"]).columns.tolist()
cat_cols = X_clf_train.select_dtypes(include=["str", "object"]).columns.tolist()

print("Числовые:", len(num_cols), "| категориальные:", cat_cols)

# Числовые: медиана (устойчива к выбросам ПР-2) + масштабирование
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Категориальные: пропуски -> "unknown", затем one-hot
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols),
])

Числовые: 27 | категориальные: ['neo', 'class']


In [5]:
# Обучаем препроцессор ТОЛЬКО на train — это и есть защита от утечки
preprocessor.fit(X_clf_train)

X_clf_train_prep = preprocessor.transform(X_clf_train)
X_clf_test_prep = preprocessor.transform(X_clf_test)

print("Train после препроцессинга:", X_clf_train_prep.shape)
print("Test после препроцессинга: ", X_clf_test_prep.shape)

Train после препроцессинга: (750882, 43)
Test после препроцессинга:  (187721, 43)


In [6]:
# Имена итоговых признаков (one-hot раскрыл категории)
feature_names = preprocessor.get_feature_names_out()
print("Всего признаков после one-hot:", len(feature_names))
print("Первые 10:", feature_names[:10])
print("Категориальные:", [f for f in feature_names if f.startswith("cat_")])

# Контроль утечки: медиана посчитана по train и применена к test.
# Если бы мы заполняли пропуски ДО разбиения — test влиял бы на train.
print("\nМедиана moid (train):", X_clf_train["moid"].median())
print("Медиана moid (test): ", X_clf_test["moid"].median())
print("Совпадение не требуется — imputer использовал только train-значение.")

Всего признаков после one-hot: 43
Первые 10: ['num__H' 'num__albedo' 'num__epoch' 'num__e' 'num__a' 'num__q' 'num__i'
 'num__om' 'num__w' 'num__ma']
Категориальные: ['cat__neo_N', 'cat__neo_Y', 'cat__neo_unknown', 'cat__class_AMO', 'cat__class_APO', 'cat__class_AST', 'cat__class_ATE', 'cat__class_CEN', 'cat__class_HYA', 'cat__class_IEO', 'cat__class_IMB', 'cat__class_MBA', 'cat__class_MCA', 'cat__class_OMB', 'cat__class_TJN', 'cat__class_TNO']

Медиана moid (train): 1.24105
Медиана moid (test):  1.23992
Совпадение не требуется — imputer использовал только train-значение.


## Вывод по ПР-3

1. **Отбор признаков:** удалены ID и служебные поля, дубли (corr ≈ 1 из ПР-2),
   константа `equinox`, `diameter_sigma` (погрешность целевой — прокси-утечка).
2. **Две задачи:** классификация `pha` (938 603 объектов, diameter исключён из-за
   86% пропусков) и регрессия `diameter` (136 209 объектов с известным диаметром).
3. **Разбиение до статистик** с `stratify` для классификации (защита от потери
   класса Y в test при дисбалансе 0.22%).
4. **Стратегии заполнения:** числовые — медиана (устойчива к выбросам ПР-2),
   категориальные — константа "unknown".
5. **Утечка отсутствует:** препроцессор обучен (`fit`) только на train;
   test только трансформируется. Размерности train/test согласованы.